# Ground truth for the team: 30 Dutch questions → CSV

Builds `ground_truth_nl_30.csv` from the Dutch ground-truth files in `evaluation/ground_truth/`.

**Columns**
| Column | Content |
|---|---|
| `id` | Original id (S01, T04, C06, UC1-01, …), so each row traces back to its source |
| `question` | The question (Dutch) |
| `ground_truth` | The correct answer (Dutch) |
| `context` | The evidence: a **JSON list of Dutch quotes** from the policy, e.g. `["quote 1", "quote 2"]`. `[]` when the policy has no answer |
| `level` | `easy`, `medium` or `hard` |
| `source` | Where each quote is: a **JSON list** like `["p. 8 · 1.1 Algemene regels"]`, **same order as `context`** (quote *i* is at location *i*). The page is the physical PDF page (1–95) a PDF viewer shows, not the label printed on the page |

It also builds `ground_truth_nl_30_review.pdf` for the reviewer: per question a cover page (question, answer, quotes
with locations) followed by the cited pages of the policy with each quote **highlighted in yellow**.

**Levels**
- **easy**: the answer is in one passage (single-hop + use-case fact lookups)
- **medium**: needs 2–5 passages, or a calculation (two-hop, multi-hop, use-case calculations)
- **hard**: aggregation over many places, traps, full case checks, and questions the policy can't answer

**Status:** the answers are DRAFT (not yet reviewed by a mortgage expert). The use-case questions (UC…) were used to
design the ontology rules, so they favour that approach.

In [ ]:
import json
from pathlib import Path

import pandas as pd

# Change these if the notebook is not in <repo>/notebooks/
GT_DIR = Path("../evaluation/ground_truth")
OUT_FILE = GT_DIR / "csv" / "ground_truth_nl_30.csv"
REVIEW_PDF = GT_DIR / "csv" / "ground_truth_nl_30_review.pdf"
POLICY_PDF = Path("../resources/pdfs/rabobank_mortgage.pdf")
COLUMNS = ["id", "question", "ground_truth", "context", "level", "source"]

SOURCES = ["qa_rabobank_nl.json", "qa_usecases_nl.json"]

## 1. The 30 picks

10 per level, spread over all chapters of the policy (applicant, income, obligations, loan, collateral, Explain,
Beheerbeleid). Reasons for each pick are in the comments. To swap a question, change its id here.

In [ ]:
PICKS = {
    "easy": [
        "S01",     # maximum term 30 years (1.1)
        "S02",     # applicant: natural person, 18+ (2.1)
        "S03",     # 3-5 debtors: channel and conditions (2.2)
        "S05",     # acceptable RRR 1-16 (2.4.2)
        "S06",     # AOW age 67 years 3 months (ch. 3)
        "S07",     # DUO grossing-up factor 1.40 (4.7)
        "S09",     # diplomat clause, max 3 years (Beheer 4.1.1)
        "S10",     # revaluation above 3M every 3 years (Beheer 1.6)
        "UC1-06",  # who values a houseboat, NWWI not required (5.5.12)
        "UC8-06",  # may a holiday home be let (5.5.13)
    ],
    "medium": [
        "T04",     # woonwagen only with NHG; NHG rules win (5.5.10 + 1.1.2)
        "T05",     # 4 applicants + EVA register (2.2 + 2.4.1)
        "T06",     # single applicant, senior within 10 years (6.1.1.1 + 6.1.1.2)
        "T08",     # shop with home: 100% / 80% + no desktop (5.5.4 + 5.4)
        "M03",     # divorce + NHG, release from joint liability (6.1.1 + Beheer 3.1)
        "M05",     # bridging loan + double costs + construction deposit (1.4.x)
        "M07",     # farmhouse > 3 ha, business use (5.5.6 -> 5.5.8 + 5.4)
        "M10",     # municipal ground lease Amsterdam 2016 (5.2.1 + annex 1 + 4.9.1 + 5.4)
        "UC2-04",  # calculation: max interest-only on EUR 400,000 (1.1.3)
        "UC4-01",  # calculation: energy measures, 106% cap (1.3 + 5.5.2)
    ],
    "hard": [
        "C01",     # all collateral types without interest-only (4+ sections)
        "C06",     # two look-alike energy-label tables (1.2.2)
        "C07",     # every way above 100%, each with its own cap (6 places)
        "C10",     # tailored solutions combinable with NHG (conditions/exceptions)
        "UC1-01",  # houseboat, 4 violations -> reject
        "UC2-01",  # unpledged savings mortgage = hidden interest-only -> reject
        "UC7-01",  # father + son (17) + friend, 6 sections -> reject
        "UC9-01",  # woonark + RRR R17 -> Explain (not reject)
        "UC1-08",  # unanswerable: interest rate is not in the policy
        "UC8-08",  # unanswerable: a fact (zoning) is missing
    ],
}

## 2. Build the table

In [ ]:
questions = {}
for name in SOURCES:
    data = json.loads((GT_DIR / name).read_text(encoding="utf-8"))
    assert data["metadata"]["language"] == "nl", f"{name} is not the Dutch file"
    for q in data["questions"]:
        assert q["id"] not in questions, f"duplicate id {q['id']}"
        questions[q["id"]] = q

rows = []
for level, ids in PICKS.items():
    for qid in ids:
        q = questions[qid]  # KeyError here = a typo in PICKS
        rows.append({
            "id": qid,
            "question": q["question"],
            "ground_truth": q["answer"],
            "context": json.dumps([e["quote"] for e in q["evidence"]], ensure_ascii=False),
            "level": level,
            "source": json.dumps([f"p. {e['page']} · {e['section']}" for e in q["evidence"]], ensure_ascii=False),
        })

gt = pd.DataFrame(rows, columns=COLUMNS)
gt["level"].value_counts()

## 3. Checks

In [ ]:
all_ids = [i for ids in PICKS.values() for i in ids]
assert len(all_ids) == 30 and len(set(all_ids)) == 30, "need 30 unique ids"
assert gt["level"].value_counts().to_dict() == {"easy": 10, "medium": 10, "hard": 10}
assert (gt["question"].str.strip() != "").all() and (gt["ground_truth"].str.strip() != "").all()

# context parses back to a list of strings; only unanswerable questions may have none
for _, r in gt.iterrows():
    ctx = json.loads(r["context"])
    assert isinstance(ctx, list) and all(isinstance(c, str) and c for c in ctx), r["id"]
    if not ctx:
        assert questions[r["id"]]["category"] == "unanswerable", f"{r['id']} has no evidence"
    src = json.loads(r["source"])
    assert len(src) == len(ctx), f"{r['id']}: source and context differ in length"
    assert all(1 <= int(s.split()[1]) <= 95 for s in src), f"{r['id']}: page outside 1-95"

print("All checks passed.")
gt.assign(n_quotes=gt["context"].map(lambda c: len(json.loads(c))))[["id", "level", "n_quotes", "question"]]

## 4. Write the CSV

UTF-8 with BOM (`utf-8-sig`), so Excel shows €, ë and é correctly.

In [ ]:
OUT_FILE.parent.mkdir(parents=True, exist_ok=True)
gt.to_csv(OUT_FILE, index=False, encoding="utf-8-sig")

# Round trip: what the team reads back must equal what we wrote
back = pd.read_csv(OUT_FILE, encoding="utf-8-sig", keep_default_na=False)
assert list(back.columns) == COLUMNS
assert back.equals(gt)
print(f"written {OUT_FILE} ({len(back)} rows)")

## 5. Review PDF: the cited pages with the quotes highlighted

For each question: a cover page (id, level, question, answer, the numbered quotes with their location), then the
cited policy pages with the quotes highlighted in yellow. The bookmarks in the PDF jump to each question.

Quotes were lightly cleaned when the ground truth was written (hyphenation joined, € restored, table rows written
as "label value"), so a quote is not always found word for word. Then the search falls back to pieces of 4 words
and highlights the pieces it finds. The cover page says per quote how it was found: **full**, **partial**, or
**not found** (look it up by hand on the given page).

In [ ]:
import html

import pymupdf

YELLOW = (1, 0.92, 0.2)


def find_table_row(page, quote):
    # A table quote is "row label value" (e.g. "C, D 5.000"); in the PDF they are separate cells.
    # Find the label, then the cell with the value in the same row (closest one to the right).
    label, _, value = quote.rpartition(" ")
    if not label or not any(ch.isdigit() for ch in value):
        return []
    labels = []
    for candidate in (label, label.rstrip("- ")):  # a label can end in a hyphen where the PDF wraps the word
        labels = page.search_for(candidate)
        if labels:
            break
    # the value must be a whole word ("0" must not match inside "40.000")
    values = [pymupdf.Rect(w[:4]) for w in page.get_text("words") if w[4] == value]
    best = None
    for lr in labels:
        for vr in values:
            same_row = abs((vr.y0 + vr.y1) / 2 - (lr.y0 + lr.y1) / 2) < 6
            if same_row and vr.x0 > lr.x0:
                gap = vr.x0 - lr.x1
                if best is None or gap < best[0]:
                    best = (gap, lr, vr)
    return [best[1].quad, best[2].quad] if best else []


def reading_key(page, rect):
    # The policy pages have 3 columns: read column by column, top to bottom.
    return (int(rect.x0 // (page.rect.width / 3)), rect.y0)


def pick_in_section(page, quads, quote, section):
    # A short sentence can occur several times on a page (e.g. "Aflossingsvrij lenen is niet toegestaan." under
    # three collateral types). Keep the occurrence right after the cited section's heading. Quotes cited for
    # several sections at once ("5.5.7 / 5.5.8") keep all occurrences.
    n = " ".join(page.get_text().split()).count(" ".join(quote.split()))
    if n <= 1 or "/" in section or len(quads) % n:
        return quads
    per = len(quads) // n
    occurrences = [quads[i:i + per] for i in range(0, len(quads), per)]
    heading = page.search_for(section.split()[0] + " ")
    if not heading:  # the section starts on an earlier page: its text comes first on this page
        return occurrences[0]
    start = reading_key(page, heading[0])
    after = [o for o in occurrences if reading_key(page, o[0].rect) > start]
    return after[0] if after else occurrences[0]


def find_quote(page, quote, section=""):
    # Return (quads, how): the whole quote if found, else a table row, else 4-word pieces.
    quads = page.search_for(quote, quads=True)
    if quads:
        return pick_in_section(page, quads, quote, section), "full"
    if len(quote.split()) <= 8:
        quads = find_table_row(page, quote)
        if quads:
            return quads, "table row"
    found = find_pieces(page, quote)
    return (found, "partial") if found else ([], "not found")


def find_pieces(page, quote, size=4, max_gap=60):
    # Search the quote in pieces of `size` words. A short piece can occur in several places, so keep one chain:
    # each next piece must come after the previous one in reading order (same column, at most `max_gap` pt lower).
    words = quote.split()
    pieces = [" ".join(words[i:i + size]) for i in range(0, len(words), size)]
    hits = [page.search_for(p, quads=True) if len(p) >= 8 else [] for p in pieces]

    def follows(prev, q):
        same_column = abs(q.rect.x0 - prev.rect.x0) < 200
        below = 0 <= q.rect.y0 - prev.rect.y0 <= max_gap or (abs(q.rect.y0 - prev.rect.y0) < 3 and q.rect.x0 >= prev.rect.x1 - 1)
        return same_column and below

    best = []
    for i, start_hits in enumerate(hits):
        for start in start_hits:
            chain, prev = [start], start
            for later in hits[i + 1:]:
                nxt = next((q for q in later if follows(prev, q)), None)
                if nxt is not None:
                    chain.append(nxt)
                    prev = nxt
            if len(chain) > len(best):
                best = chain
    return best if len(best) >= max(2, len(pieces) // 2) else []


def cover_html(r, q, results):
    items = "".join(
        f"<li><b>{html.escape(src)}</b> <i>[{how}]</i><br>{html.escape(e['quote'])}</li>"
        for e, src, how in results)
    if not items:
        items = "<li><i>No evidence: the policy does not answer this question.</i></li>"
    return (f"<h2>{r['id']} · {r['level']}</h2>"
            f"<p><b>Vraag:</b> {html.escape(r['question'])}</p>"
            f"<p><b>Antwoord (ground truth):</b> {html.escape(r['ground_truth'])}</p>"
            f"<p><b>Bewijs</b> (cited pages follow, quotes highlighted):</p><ol>{items}</ol>")


review = pymupdf.open()
toc, report = [], []
for _, r in gt.iterrows():
    q = questions[r["id"]]
    src = pymupdf.open(POLICY_PDF)  # fresh copy per question, so highlights don't mix between questions
    results, pages = [], []
    for e, loc in zip(q["evidence"], json.loads(r["source"])):
        page = src[e["page"] - 1]
        quads, how = find_quote(page, e["quote"], e["section"])
        for quad in quads:
            page.add_highlight_annot(quad).set_colors(stroke=YELLOW)
        results.append((e, loc, how))
        report.append({"id": r["id"], "source": loc, "found": how})
        if e["page"] not in pages:
            pages.append(e["page"])

    cover = review.new_page(width=842, height=595)  # A4 landscape, like the policy
    cover.insert_htmlbox(pymupdf.Rect(40, 40, 802, 555), cover_html(r, q, results),
                         css="* {font-family: sans-serif; font-size: 10pt;} h2 {font-size: 15pt;}")
    toc.append([1, f"{r['id']} ({r['level']})", review.page_count])
    for p in pages:
        review.insert_pdf(src, from_page=p - 1, to_page=p - 1)
        toc.append([2, f"p. {p}", review.page_count])
    src.close()

review.set_toc(toc)
review.save(REVIEW_PDF, garbage=3, deflate=True)
print(f"written {REVIEW_PDF} ({review.page_count} pages)")

found = pd.DataFrame(report)
print(found["found"].value_counts().to_string())
found[found["found"] != "full"]

## Reading it in the team repo

```python
import json
import pandas as pd

gt = pd.read_csv("ground_truth_nl_30.csv", encoding="utf-8-sig", keep_default_na=False)
gt["context"] = gt["context"].apply(json.loads)   # back to a list of quotes per question
gt["source"] = gt["source"].apply(json.loads)     # the location of each quote, same order
```

If the team wants only the five original columns, drop `source`:
`pd.read_csv(..., usecols=["id", "question", "ground_truth", "context", "level"])`.

Two questions are unanswerable on purpose: **UC1-08** (not in the policy, `context` is `[]`) and **UC8-08** (a fact
is missing; `context` holds the rule that needs it). A correct answer says it can't decide and why. Make sure the
judge or metric doesn't score that as wrong.